In [31]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
df = pd.read_csv("Base_de_Transacoes_e_Cupons_Capturados.csv", sep=";")

df

,celular,data,hora,nome_estabelecimento,bairro_estabelecimento,categoria_estabelecimento,id_campanha,id_cupom,tipo_cupom,produto,valor_cupom,repasse_picmoney
0,(61) 96497-8673,10/07/2025,16:15:00,Habib's,República,Lojas de Eletrônicos e Games,CAM2768,CUP542835,Cashback,NaN,229.64,11.48
1,(11) 94231-6424,15/07/2025,08:15:00,Smart Fit,Vila Prudente,Lojas de Eletrônicos e Games,CAM6679,CUP291620,Cashback,NaN,356.33,17.82
2,(11) 97965-2178,20/07/2025,16:45:00,Outback,Tucuruvi,Igrejas e Lojas de Artigos Religiosos,CAM6473,CUP670811,Produto,Tempora,719.06,27.61
3,(11) 93418-4646,20/07/2025,15:45:00,Subway,Penha,Fisioterapia e Terapias Complementares,CAM8293,CUP590364,Produto,Magnam,798.34,25.85
4,(11) 97973-1725,07/07/2025,11:00:00,Octavio Café,Santo Amaro,Clínicas Médicas e Laboratórios,CAM5588,CUP528033,Produto,Quam,718.45,28.85
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,(21) 95319-5062,27/07/2025,09:00:00,Droga Raia,Penha,Farmácias e Drogarias,CAM1614,CUP400268,Desconto,NaN,863.65,310.22
99996,(11) 92043-7072,05/07/2025,06:45:00,Clube Pinheiros,Ipiranga,Clínicas Médicas e Laboratórios,CAM7293,CUP137216,Desconto,NaN,251.03,64.32
99997,(11) 96610-1439,06/07/2025,15:45:00,Outback,Vila Prudente,Lojas de Eletrônicos e Games,CAM3065,CUP551535,Cashback,NaN,160.48,8.02
99998,(11) 92639-5993,02/07/2025,18:45:00,Açaí no Ponto,Jabaquara,"Bancos, Agências e Correspondentes",CAM3552,CUP390792,Desconto,NaN,665.09,124.46


In [32]:

receita_total = df["valor_cupom"].sum()

total_repasses = df["repasse_picmoney"].sum()

receita_liquida = receita_total - total_repasses

volume_transacoes = len(df)

ticket_medio = receita_total / volume_transacoes

margem_operacional = (receita_liquida / receita_total) * 100

dist_cupom = df["tipo_cupom"].value_counts(normalize=True) * 100

receita_categoria = df.groupby("categoria_estabelecimento")["valor_cupom"].sum()

df["data"] = pd.to_datetime(df["data"], dayfirst=True)
receita_semana = df.groupby(df["data"].dt.day_name())["valor_cupom"].sum()

df["hora"] = pd.to_datetime(df["hora"], format="%H:%M:%S").dt.hour
df["periodo"] = pd.cut(df["hora"],
                       bins=[0,6,12,18,24],
                       labels=["Madrugada","Manhã","Tarde","Noite"],
                       right=False)
receita_periodo = df.groupby("periodo")["valor_cupom"].sum()

C:\Users\24025889\AppData\Local\Temp\ipykernel_27280\1201518339.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  receita_periodo = df.groupby("periodo")["valor_cupom"].sum()


In [33]:
# garante que a coluna existe
df["categoria_estabelecimento"] = df["nome_estabelecimento"]

# percorre cada linha e ajusta a categoria
novas_categorias = []

for nome in df["nome_estabelecimento"]:
    match nome:
        # Restaurantes e Lanchonetes
        case "Habib's" | "Outback" | "Subway" | "Octavio Café" | "Madero" | "Starbucks" | "Café Cultura" | "Burger King" | "Açaí no Ponto" | "Churrascaria Boi Preto" | "McDonald's" | "Ráscal":
            novas_categorias.append("Restaurantes e Lanchonetes")

        # Academias e Esporte
        case "Smart Fit" | "Selfit" | "Just Run" | "Clube Pinheiros" | "Sesc Paulista" | "Sesc Carmo":
            novas_categorias.append("Academias e Esporte")

        # Moda e Varejo
        case "Forever 21" | "Renner" | "Riachuelo" | "Casas Bahia" | "Magazine Luiza" | "Ponto":
            novas_categorias.append("Moda e Varejo")

        # Supermercados
        case "Extra" | "Carrefour Express" | "Pão de Açúcar":
            novas_categorias.append("Supermercados e Hipermercados")

        # Farmácias
        case "Droga Raia" | "Drogasil" | "Drogaria São Paulo":
            novas_categorias.append("Farmácias e Drogarias")

        # Saúde
        case "Sabin" | "Lavoisier" | "Fleury":
            novas_categorias.append("Saúde e Exames")

        # caso não encontrado
        case _:
            novas_categorias.append("Outros")

# substituir na coluna
df["categoria_estabelecimento"] = novas_categorias

# verificar resultado
print(df[["nome_estabelecimento", "categoria_estabelecimento"]])


      nome_estabelecimento   categoria_estabelecimento
0                  Habib's  Restaurantes e Lanchonetes
1                Smart Fit         Academias e Esporte
2                  Outback  Restaurantes e Lanchonetes
3                   Subway  Restaurantes e Lanchonetes
4             Octavio Café  Restaurantes e Lanchonetes
...                    ...                         ...
99995           Droga Raia       Farmácias e Drogarias
99996      Clube Pinheiros         Academias e Esporte
99997              Outback  Restaurantes e Lanchonetes
99998        Açaí no Ponto  Restaurantes e Lanchonetes
99999            Lavoisier              Saúde e Exames

[100000 rows x 2 columns]


In [34]:

SAIDA_IMG = Path("g1_pizza_tipos.png")

dist = df["tipo_cupom"].value_counts(normalize=True) * 100

plt.figure()
plt.pie(dist.values, labels=dist.index, autopct="%1.1f%%")
plt.title("Distribuição por Tipo de Cupom (% do volume)")
plt.tight_layout()
plt.savefig(SAIDA_IMG, dpi=150)
plt.close()


In [35]:
SAIDA_IMG = Path("g2_top5_categorias.png")

top5 = (
    df.groupby("categoria_estabelecimento")["valor_cupom"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

plt.figure()
plt.bar(top5.index, top5.values)
plt.title("Top 5 Categorias por Receita")
plt.xticks(rotation=30, ha="right")
plt.ylabel("Receita (R$)")
plt.tight_layout()
plt.savefig(SAIDA_IMG, dpi=150)
plt.close()


In [36]:
SAIDA_IMG = Path("g3_receita_dia_semana.png")

df["data"] = pd.to_datetime(df["data"], dayfirst=True)
receita_semana = df.groupby(df["data"].dt.day_name())["valor_cupom"].sum()

ordem = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
mapa = {"Monday":"Segunda","Tuesday":"Terça","Wednesday":"Quarta",
        "Thursday":"Quinta","Friday":"Sexta","Saturday":"Sábado","Sunday":"Domingo"}

receita_semana = receita_semana.reindex(ordem).rename(index=mapa)

plt.figure()
plt.bar(receita_semana.index, receita_semana.values)
plt.title("Receita por Dia da Semana")
plt.ylabel("Receita (R$)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(SAIDA_IMG, dpi=150)
plt.close()


In [37]:
SAIDA_IMG = Path("g4_receita_periodo.png")

df["hora"] = pd.to_datetime(df["hora"], format="%H:%M:%S").dt.hour
bins = [0,6,12,18,24]
labels = ["Madrugada","Manhã","Tarde","Noite"]
df["periodo"] = pd.cut(df["hora"], bins=bins, labels=labels, right=False)

receita_periodo = df.groupby("periodo")["valor_cupom"].sum().reindex(labels)

plt.figure()
plt.bar(receita_periodo.index.astype(str), receita_periodo.values)
plt.title("Receita por Período do Dia")
plt.ylabel("Receita (R$)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(SAIDA_IMG, dpi=150)
plt.close()


ValueError: time data "16" doesn't match format "%H:%M:%S", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [12]:
SAIDA_IMG = Path("g5_kpis.png")

receita_total      = df["valor_cupom"].sum()
total_repasses     = df["repasse_picmoney"].sum()
receita_liquida    = receita_total - total_repasses
volume_transacoes  = len(df)
ticket_medio       = receita_total / volume_transacoes
margem_operacional = (receita_liquida / receita_total) * 100

kpis = [
    ("Receita Total",       f"R$ {receita_total:,.2f}"),
    ("Total de Repasses",   f"R$ {total_repasses:,.2f}"),
    ("Receita Líquida",     f"R$ {receita_liquida:,.2f}"),
    ("Volume Transações",   f"{volume_transacoes:,}"),
    ("Ticket Médio",        f"R$ {ticket_medio:,.2f}"),
    ("Margem Operacional",  f"{margem_operacional:.2f}%"),
]

fig = plt.figure(figsize=(10, 6))
fig.suptitle("KPIs Financeiros", fontsize=16)

for i, (titulo, valor) in enumerate(kpis, start=1):
    ax = fig.add_subplot(2, 3, i)
    ax.axis("off")
    ax.text(0.5, 0.65, titulo, ha="center", va="center", fontsize=11, weight="bold")
    ax.text(0.5, 0.35, valor,  ha="center", va="center", fontsize=13)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(SAIDA_IMG, dpi=150)
plt.close()
